# Kh?m ph? d? li?u WESAD (EDA)

Notebook n?y ki?m tra b?ng ??c tr?ng ?? tr?ch xu?t t? c?m bi?n ng?c WESAD tr??c khi x?y d?ng m? h?nh. Ph?m vi hi?n t?i g?m ECG, EDA v? Resp; m?i m?u l? m?t c?a s? 60 gi?y, b??c 60 gi?y. C?c nh?n ???c gi? l?i l? Baseline (`1`), Stress (`2`) v? Amusement (`3`).

D? li?u ??u v?o l? `all_subjects.csv` trong th? m?c ??c tr?ng `chest_60s_step60_raw_stats_v1`. C?c ??c tr?ng l? th?ng k? tr?n t?n hi?u th?, ch?a ph?i HRV, SCR hay nh?p th? ?? x? l? chuy?n bi?t.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Tìm thư mục gốc khi mở notebook từ gốc dự án hoặc notebooks/
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

FEATURE_DIR = (
    PROJECT_ROOT / "data" / "features"
    / "chest_60s_step60_raw_stats_v1"
)

df = pd.read_csv(FEATURE_DIR / "all_subjects.csv")

print("Số dòng, số cột:", df.shape)
print("\nDanh sách cột:")
print(df.columns.tolist())

display(df.head())
df.info()

Số dòng, số cột: (513, 27)

Danh sách cột:
['subject_id', 'window_id', 'start_s', 'end_s', 'label_original', 'quality_flag', 'ecg_mean', 'ecg_std', 'ecg_min', 'ecg_max', 'ecg_rms', 'ecg_iqr', 'ecg_range', 'eda_mean', 'eda_std', 'eda_min', 'eda_max', 'eda_rms', 'eda_iqr', 'eda_range', 'resp_mean', 'resp_std', 'resp_min', 'resp_max', 'resp_rms', 'resp_iqr', 'resp_range']


,subject_id,window_id,start_s,end_s,label_original,quality_flag,ecg_mean,ecg_std,ecg_min,ecg_max,...,eda_rms,eda_iqr,eda_range,resp_mean,resp_std,resp_min,resp_max,resp_rms,resp_iqr,resp_range
0,S2,S2_252000,360.0,420.0,1,finite_only_not_physiological_QA,0.000986,0.153720,-0.302902,0.862701,...,4.193222,0.410843,0.899887,0.061072,1.595619,-3.147888,3.730774,1.596787,2.864456,6.878662
1,S2,S2_294000,420.0,480.0,1,finite_only_not_physiological_QA,0.001455,0.182025,-1.499405,1.415726,...,3.551126,0.291443,0.663757,0.033488,1.885184,-6.646729,5.216980,1.885482,3.097916,11.863708
2,S2,S2_336000,480.0,540.0,1,finite_only_not_physiological_QA,0.001022,0.146379,-0.404984,0.776138,...,3.087097,0.274277,0.628281,0.073635,1.537771,-2.944946,4.554749,1.539533,2.726746,7.499695
3,S2,S2_378000,540.0,600.0,1,finite_only_not_physiological_QA,0.001122,0.144752,-0.272415,0.778793,...,2.625256,0.185108,0.689316,0.062645,1.441556,-3.240967,3.205872,1.442916,2.655029,6.446838
4,S2,S2_420000,600.0,660.0,1,finite_only_not_physiological_QA,0.001076,0.143769,-0.280563,0.806442,...,2.307896,0.134277,0.423813,0.038453,1.469767,-2.873230,3.224182,1.470270,2.658081,6.097412


<class 'pandas.DataFrame'>
RangeIndex: 513 entries, 0 to 512
Data columns (total 27 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   subject_id      513 non-null    str    
 1   window_id       513 non-null    str    
 2   start_s         513 non-null    float64
 3   end_s           513 non-null    float64
 4   label_original  513 non-null    int64  
 5   quality_flag    513 non-null    str    
 6   ecg_mean        513 non-null    float64
 7   ecg_std         513 non-null    float64
 8   ecg_min         513 non-null    float64
 9   ecg_max         513 non-null    float64
 10  ecg_rms         513 non-null    float64
 11  ecg_iqr         513 non-null    float64
 12  ecg_range       513 non-null    float64
 13  eda_mean        513 non-null    float64
 14  eda_std         513 non-null    float64
 15  eda_min         513 non-null    float64
 16  eda_max         513 non-null    float64
 17  eda_rms         513 non-null    float64
 18  e

## 1. N?p d? li?u v? xem c?u tr?c

Cell ti?p theo x?c ??nh th? m?c d? ?n, ??c b?ng t?ng h?p v? hi?n th? k?ch th??c, t?n c?t, m?t v?i d?ng ??u c?ng ki?u d? li?u. C?c c?t qu?n l? gi?p truy v?t c?a s? v? ng??i tham gia; kh?ng d?ng ch?ng l?m ??c tr?ng d? ?o?n.


In [ ]:
feature_cols = [
    col for col in df.columns
    if col.startswith(("ecg_", "eda_", "resp_"))
]

print("Số cửa sổ:", len(df))
print("Số người:", df["subject_id"].nunique())
print("Danh sách người:", sorted(df["subject_id"].unique()))
print("Số đặc trưng:", len(feature_cols))

print("\nPhân bố nhãn:")
display(
    df["label_original"]
    .value_counts()
    .sort_index()
    .rename_axis("label")
    .to_frame("so_cua_so")
)

print("\nSố cửa sổ theo người và nhãn:")
display(pd.crosstab(df["subject_id"], df["label_original"]))

print("\nKiểm tra chất lượng:")
print("Tổng giá trị thiếu:", df.isna().sum().sum())
print(
    "Tổng giá trị vô cực trong đặc trưng:",
    np.isinf(df[feature_cols].to_numpy()).sum()
)
print(
    "Số cửa sổ trùng theo người và mã cửa sổ:",
    df.duplicated(subset=["subject_id", "window_id"]).sum()
)
print(
    "Tất cả cửa sổ đều dài 60 giây:",
    np.isclose(df["end_s"] - df["start_s"], 60).all()
)

print("\nCác cờ chất lượng:")
display(df["quality_flag"].value_counts(dropna=False))

print("\nThống kê đặc trưng:")
display(df[feature_cols].describe().T)

Số cửa sổ: 513
Số người: 15
Danh sách người: ['S10', 'S11', 'S13', 'S14', 'S15', 'S16', 'S17', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9']
Số đặc trưng: 21

Phân bố nhãn:


,so_cua_so
label,
1,282
2,152
3,79



Số cửa sổ theo người và nhãn:


label_original,1,2,3
subject_id,,,
S10,19,11,5
S11,19,11,5
S13,19,10,6
S14,19,10,6
S15,19,11,6
S16,19,10,5
S17,19,11,5
S2,18,10,5
S3,18,10,5



Kiểm tra chất lượng:
Tổng giá trị thiếu: 0
Tổng giá trị vô cực trong đặc trưng: 0
Số cửa sổ trùng theo người và mã cửa sổ: 0
Tất cả cửa sổ đều dài 60 giây: True

Các cờ chất lượng:


quality_flag
finite_only_not_physiological_QA    513
Name: count, dtype: int64


Thống kê đặc trưng:


,count,mean,std,min,25%,50%,75%,max
ecg_mean,513.0,0.001097,0.000470,-0.001972,0.000878,0.001125,0.001372,0.002325
ecg_std,513.0,0.262983,0.064003,0.109710,0.230189,0.267879,0.299357,0.476084
ecg_min,513.0,-0.633059,0.246622,-1.500000,-0.775726,-0.573807,-0.442703,-0.213089
ecg_max,513.0,1.426927,0.189003,0.769455,1.496658,1.499222,1.499451,1.499954
ecg_rms,513.0,0.262986,0.064002,0.109718,0.230194,0.267881,0.299361,0.476085
ecg_iqr,513.0,0.136166,0.068280,0.047333,0.092789,0.130245,0.156006,0.520947
ecg_range,513.0,2.059986,0.339442,1.043427,1.920593,2.035950,2.255585,2.999451
eda_mean,513.0,4.589126,3.463852,0.635997,1.975580,3.708698,6.308028,19.969665
eda_std,513.0,0.051785,0.085745,0.007825,0.013666,0.018298,0.045203,1.129439
eda_min,513.0,4.345533,3.429493,0.000000,1.697922,3.504944,6.077194,19.680405


## 2. Ph?n b? m?u v? ki?m tra ch?t l??ng

Cell d??i ??y ??m c?a s? theo nh?n v? ng??i tham gia, sau ?? ki?m tra gi? tr? thi?u/v? c?c, c?a s? tr?ng l?p, ?? d?i c?a s? v? `quality_flag`. H?y xem b?ng ch?o ?? nh?n bi?t m?t c?n b?ng gi?a l?p ho?c kh?c bi?t v? s? m?u gi?a ng??i tham gia.

`quality_flag` ch? ph?n ?nh c?c quy t?c ki?m tra ???c tri?n khai; n? kh?ng ??m b?o t?n hi?u ??t ch?t l??ng sinh l?. N?u c? gi? tr? thi?u ho?c v? c?c, c?n t?m nguy?n nh?n tr??c khi quy?t ??nh x? l?.


## 3. Th?ng k? m? t? c?c ??c tr?ng

B?ng `describe()` t?m t?t ph?n b? t?ng ??c tr?ng tr?n to?n b? d? li?u. L?u ? c?c t?n hi?u c? th? kh?c thang ?o, v? th?ng k? g?p c? th? che khu?t kh?c bi?t gi?a nh?n ho?c gi?a ng??i tham gia. C?c b??c ti?p theo c? th? tr?c quan h?a ph?n b? theo nh?n v? ki?m tra t??ng quan.


## 4. Ghi ch? cho b??c m? h?nh h?a

- D?ng 21 c?t ??c tr?ng c? ti?n t? `ecg_`, `eda_`, `resp_` l?m `X`; d?ng `label_original` l?m `y`.
- Kh?ng ??a `subject_id`, `window_id`, `start_s`, `end_s` ho?c `quality_flag` v?o `X`.
- Chia train/test theo ng??i tham gia ?? tr?nh c?a s? c?a c?ng m?t ng??i xu?t hi?n ? c? hai t?p.
- Fit imputer, scaler v? b??c ch?n ??c tr?ng b?n trong t?ng fold, ch? tr?n ph?n train.

Notebook n?y m? t? d? li?u; quan s?t kh?c bi?t gi?a c?c l?p ch?a ?? ?? k?t lu?n m? h?nh c? th? kh?i qu?t t?t.
